# Analyse du dernier `tuning_log_*.csv`

1. Timing de la boucle (fréquence réelle, répartition détection / estimation / contrôle / actuation).
2. Si le CSV contient la position de la balle et les commandes (`x_mm`, `y_mm`, `vx`, `vy`, `roll`, `pitch`), analyse de la réponse du PID.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Les noms contiennent la date au format YYYY-MM-DD_HH-MM-SS, donc le tri
# alphabétique = tri chronologique. Remplacer par un chemin pour un autre run.
LOG_PATH = sorted(Path('.').glob('tuning_log_*.csv'))[-1]

# Les premiers ticks incluent le démarrage caméra (~0.7 s) : on les ignore.
SKIP_TICKS = 5

with open(LOG_PATH, newline='') as f:
    rows = list(csv.DictReader(f))
data = {k: np.array([float(r[k]) for r in rows[SKIP_TICKS:]]) for k in rows[0]}

print(f'{LOG_PATH} : {len(rows)} ticks, colonnes = {list(data)}')

## 1. Timing de la boucle

In [ ]:
dt = data['dt']
t = np.cumsum(dt)
stages = [k for k in ('dt_detection', 'dt_estimation', 'dt_control', 'dt_actuation') if k in data]

print(f'Durée du run      : {t[-1]:.1f} s')
print(f'Fréquence médiane : {1 / np.median(dt):.1f} Hz')
print(f'dt médian / p95 / max : {np.median(dt)*1e3:.1f} / {np.percentile(dt, 95)*1e3:.1f} / {dt.max()*1e3:.1f} ms')
print()
for k in stages:
    v = data[k] * 1e3
    print(f'{k:15s} médiane {np.median(v):7.3f} ms   p95 {np.percentile(v, 95):7.3f} ms   max {v.max():7.3f} ms   ({100 * v.sum() / (dt.sum() * 1e3):5.1f} % du temps)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(t, dt * 1e3, lw=0.8, label='dt total')
for k in stages:
    axes[0].plot(t, data[k] * 1e3, lw=0.8, label=k)
axes[0].set_xlabel('t (s)')
axes[0].set_ylabel('ms')
axes[0].set_title('Durée par tick')
axes[0].legend()

axes[1].hist(dt * 1e3, bins=60)
axes[1].axvline(1000 / 30, color='k', ls='--', label='1 frame @ 30 fps')
axes[1].set_xlabel('dt (ms)')
axes[1].set_title('Distribution de dt')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Réponse du PID

Nécessite que `main.py` logge `x_mm, y_mm, vx, vy, roll, pitch` (erreur en mm, sortie en rad). Sinon cette section est ignorée.

In [ ]:
PID_COLS = ('x_mm', 'y_mm', 'roll', 'pitch')
HAS_PID = all(k in data for k in PID_COLS)

if not HAS_PID:
    print('Pas de colonnes position/commande dans ce log : section ignorée.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(t, data['x_mm'], label='x (mm)')
    axes[0].plot(t, data['y_mm'], label='y (mm)')
    axes[0].axhline(0, color='k', lw=0.5)
    axes[0].set_ylabel('erreur (mm)')
    axes[0].legend()

    axes[1].plot(t, np.degrees(data['pitch']), label='pitch (°)')
    axes[1].plot(t, np.degrees(data['roll']), label='roll (°)')
    axes[1].set_ylabel('commande (°)')
    axes[1].set_xlabel('t (s)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def oscillation_stats(sig, t):
    """Période moyenne (passages par zéro montants) et amplitude RMS."""
    sig = sig - np.mean(sig)
    up = np.where((sig[:-1] < 0) & (sig[1:] >= 0))[0]
    period = np.mean(np.diff(t[up])) if len(up) > 1 else float('nan')
    return period, np.sqrt(np.mean(sig ** 2))


if HAS_PID:
    for axis, cmd in (('x_mm', 'pitch'), ('y_mm', 'roll')):
        period, rms = oscillation_stats(data[axis], t)
        sat = np.mean(np.abs(data[cmd]) >= 0.99 * np.abs(data[cmd]).max()) * 100
        print(f'{axis}: moyenne {data[axis].mean():6.1f} mm | RMS {rms:5.1f} mm | '
              f'période {period:5.2f} s | {cmd} saturé {sat:4.1f} % du temps')